## First ML Model

In [1]:
import torch
from torch_geometric.data import Data

def neo4j_to_pyg(driver):
    """Convert Neo4j graph to PyTorch Geometric format"""
    
    # Extract nodes
    node_query = """
    MATCH (b:Bridge)
    RETURN b.structure_number AS id,
           b.age AS age,
           b.adt AS adt,
           b.adtt AS adtt,
           b.adtl AS adtl,
           b.truck_pct AS truck_pct,
           b.deck_cr AS deck_cr,
           b.super_cr AS super_cr,
           b.sub_cr AS sub_cr,
           b.health_index AS hi,
           b.freeze_thaw AS freeze_thaw,
           b.snowfall_days AS snowfall_days,
           b.functional_class AS func_class
    ORDER BY id
    """
    
    with driver.session() as session:
        result = session.run(node_query)
        nodes_df = pd.DataFrame([dict(r) for r in result])
    
    # Create feature matrix
    feature_cols = [
        'age', 'adt', 'adtt', 'adtl', 'truck_pct',
        'deck_cr', 'super_cr', 'sub_cr', 'hi',
        'freeze_thaw', 'snowfall_days', 'func_class'
    ]
    
    X = torch.FloatTensor(nodes_df[feature_cols].fillna(0).values)
    
    # Standardize features
    X = (X - X.mean(dim=0)) / (X.std(dim=0) + 1e-8)
    
    # Extract edges
    edge_query = """
    MATCH (b1:Bridge)-[r:NEAR]->(b2:Bridge)
    RETURN b1.structure_number AS source,
           b2.structure_number AS target
    """
    
    with driver.session() as session:
        result = session.run(edge_query)
        edges_df = pd.DataFrame([dict(r) for r in result])
    
    # Create ID mapping
    id_to_idx = {id: idx for idx, id in enumerate(nodes_df['id'])}
    
    edge_index = torch.LongTensor([
        [id_to_idx[src] for src in edges_df['source']],
        [id_to_idx[tgt] for tgt in edges_df['target']]
    ])
    
    # Load labels
    labels_df = pd.read_csv('bridge_labels.csv')
    label_map = dict(zip(labels_df['bridge_id'], labels_df['label']))
    
    y = torch.LongTensor([
        label_map.get(id, -1) for id in nodes_df['id']
    ])
    
    # Create train/val/test masks
    labeled_mask = y != -1
    train_mask = torch.zeros(len(y), dtype=torch.bool)
    val_mask = torch.zeros(len(y), dtype=torch.bool)
    test_mask = torch.zeros(len(y), dtype=torch.bool)
    
    labeled_indices = torch.where(labeled_mask)[0]
    perm = torch.randperm(len(labeled_indices))
    
    n_train = int(0.7 * len(labeled_indices))
    n_val = int(0.15 * len(labeled_indices))
    
    train_mask[labeled_indices[perm[:n_train]]] = True
    val_mask[labeled_indices[perm[n_train:n_train+n_val]]] = True
    test_mask[labeled_indices[perm[n_train+n_val:]]] = True
    
    # Create PyG Data object
    data = Data(
        x=X,
        edge_index=edge_index,
        y=y,
        train_mask=train_mask,
        val_mask=val_mask,
        test_mask=test_mask,
        bridge_ids=nodes_df['id'].values
    )
    
    return data

# Execute
data = neo4j_to_pyg(loader.driver)
torch.save(data, 'bridge_graph.pt')
print(f"Graph: {data.num_nodes} nodes, {data.num_edges} edges")

OSError: [WinError 1114] A dynamic link library (DLL) initialization routine failed. Error loading "C:\Users\aksel\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\torch\lib\c10.dll" or one of its dependencies.